In [ ]:
# Cargamos las claves API en el codigo

from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

### Cargamos los archivos para que después el agente pueda acceder a ellos:

In [ ]:
# Cargar archivos

# Para los csv
carpeta_datos = "../datos_negocio"

clientes = pd.read_csv(f"{carpeta_datos}/clientes.csv")
citas = pd.read_csv(f"{carpeta_datos}/citas.csv")
ventas = pd.read_csv(f"{carpeta_datos}/ventas.csv")
gastos = pd.read_csv(f"{carpeta_datos}/gastos.csv")
empleados = pd.read_csv(f"{carpeta_datos}/empleados.csv")
compras = pd.read_csv(f"{carpeta_datos}/compras.csv")
servicios = pd.read_csv(f"{carpeta_datos}/servicios.csv")


# Para los txt 
carpeta_documentos = "../documentos_negocio"

with open(f"{carpeta_documentos}/negocio.txt", "r", encoding="utf-8") as f:
    negocio = f.read()

In [ ]:
# Convertir "negocio.txt" en un Document de LangChain

from langchain_core.documents import Document

with open(f"{carpeta_documentos}/negocio.txt", "r", encoding="utf-8") as f:
    negocio = f.read()

documento_negocio = Document(
    page_content=negocio,
    metadata={"fuente": "negocio.txt"}
)

In [ ]:
print(documento_negocio.page_content)

### Separamos el texto en partes más pequeñas (Splitter):

In [ ]:
# Splitter 

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents([documento_negocio])  # Ponemos corchetes porque "split_documents() espera una lista"

### Creamos una base de datos vectorial:

In [ ]:
# Crear el vectorstore

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(     # base de datos vectorial donde se guardan los embeddings.
    documents=splits,
    embedding=embeddings
)

In [ ]:
retriever = vectorstore.as_retriever()   # componente que posteriormente podrá buscar los fragmentos relevantes de negocio.txt

### Convertir ese retriever en una tool que el agente pueda utilizar:

In [ ]:
from langchain.tools import tool

@tool
def buscar_informacion_negocio(pregunta: str) -> str:
    "Busca información general sobre el negocio en sus documentos."

    documentos = retriever.invoke(pregunta)

    return "\n\n".join(
        documento.page_content
        for documento in documentos
    )

### Creamos memoria temoral:

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# La añadimos después en el Agente

### Crear Modelo y System Prompt:

In [ ]:
model = ChatOpenAI(
    model="gpt-4o-mini"
)

In [ ]:
from langchain.agents import create_agent

system_prompt = """
Eres un asistente especializado en gestión y marketing para pequeñas y medianas empresas.

Tu objetivo es ayudar al propietario de un negocio a gestionar mejor su empresa, ahorrar tiempo, mejorar la atención al cliente y aumentar sus ventas.

Puedes ayudar en las siguientes áreas:

- Atención y seguimiento de clientes.
- Gestión de citas y reservas.
- Análisis de ventas.
- Control de gastos y compras.
- Seguimiento de pagos al personal.
- Análisis de ingresos y beneficios.
- Marketing y captación de clientes.
- Fidelización de clientes.
- Detección de problemas y oportunidades.

Cuando analices información del negocio:
1. Diferencia entre datos, análisis y recomendaciones.
2. No inventes información que no tengas.
3. Si falta información, pregunta al usuario.
4. Da recomendaciones prácticas y concretas.
5. Utiliza un lenguaje claro y profesional.

Tu objetivo final es ayudar al propietario a tomar mejores decisiones basándose en los datos disponibles.
"""


### Crear el Agente:

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[buscar_informacion_negocio],   # Esta el la tool que hemos creado arriba con el retriever
    system_prompt=system_prompt,
    checkpointer=checkpointer     # Memoria a corto plazo creada arriba
)

#### Hacemos una prueba: 

In [ ]:
respuesta = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "¿Cómo se llama el negocio?"
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "cliente_1"    # Creamos una ID de Sesión para guardar ahí la conversación
        }
    }
)

In [ ]:
print(respuesta["messages"][-1].content)

### Pruebas: 

In [ ]:
from tools.reservas_tools import registrar_cliente

registrar_cliente.invoke({
    "nombre": "Cliente Prueba",
    "telefono": "611111111",
    "email": "clienteprueba@gmail.com"
})